# Capstone Two — Preprocessing and Training Data Development

## Predictive Healthcare Operations & Capacity Analytics

This notebook prepares the California hospital quarterly dataset for predictive modeling.

The prediction objective is to **predict next-quarter total hospital discharges (`DIS_TOT`) using information from the previous quarter**.

The notebook follows the Springboard Capstone Two Step 4 requirements:

1. Prepare a supervised dataset by matching each hospital's previous-quarter information with its next-quarter discharge total.
2. Review candidate features and missing values.
3. Split the data chronologically into training, validation, and testing datasets.
4. Impute missing values.
5. Encode categorical variables using indicator (dummy) variables.
6. Standardize numerical features.
7. Save the prepared datasets and preprocessing pipeline.

**Important:** All preprocessing steps are fitted using the training data only. The fitted transformations are then applied to validation and test data to prevent data leakage.

## 1. Import Libraries

Import the libraries needed for data preparation, chronological splitting, preprocessing, and saving the outputs.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Load the Processed Dataset

Load the cleaned HCAI hospital dataset created during the Data Wrangling stage.

In [ ]:
# Path to the processed HCAI hospital dataset.
DATA_PATH = Path("../data/processed/hcai_hospital_data_clean.csv")

# Load the dataset.
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

## 3. Review the Dataset

Confirm that the hospital identifier, reporting period, target variable, and candidate features are available.

In [ ]:
# Display the first five rows.
display(df.head())

# Review the required columns.
required_columns = [
    "FAC_NO",
    "YEAR_QTR",
    "DIS_TOT",
    "LIC_BEDS",
    "AVL_BEDS",
    "STF_BEDS",
    "DAY_TOT",
    "VIS_TOT",
    "TYPE_CNTRL",
    "TYPE_HOSP",
    "COUNTY_NAME",
    "HSA"
]

missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("All required columns are available.")
print("Number of facilities:", df["FAC_NO"].nunique())
print("Reporting periods:", sorted(df["YEAR_QTR"].dropna().unique()))

## 4. Prepare the Future-Quarter Prediction Dataset

Each original row represents one hospital in one quarter. To create a forecasting dataset, we match:

- **Predictor information:** hospital data from the previous quarter.
- **Target:** `DIS_TOT` from the following quarter.

For example, 2025 Q4 hospital information is matched with 2026 Q1 `DIS_TOT`.

Only consecutive quarters are matched. This prevents a hospital's data from being incorrectly paired across a missing reporting period.

The hospital identifier (`FAC_NO`) is used for matching records, but it is not included as a model feature.

In [ ]:
# Convert YEAR_QTR (for example, 20211) into a sortable quarter number.
# The resulting period number increases by one for each consecutive quarter.

def quarter_to_period(value):
    value = str(int(value))
    year = int(value[:4])
    quarter = int(value[-1])

    if quarter not in [1, 2, 3, 4]:
        raise ValueError(f"Unexpected quarter value: {value}")

    return year * 4 + (quarter - 1)


# Work with a copy so the original DataFrame remains unchanged.
df_model = df.copy()

# Remove rows without the information needed to identify the hospital or quarter.
df_model = df_model.dropna(subset=["FAC_NO", "YEAR_QTR"])

# Convert the reporting period into a numeric quarter sequence.
df_model["PERIOD_NUMBER"] = df_model["YEAR_QTR"].apply(quarter_to_period)

# Confirm that each hospital has no duplicate quarter records.
duplicate_count = df_model.duplicated(
    subset=["FAC_NO", "PERIOD_NUMBER"]
).sum()

print("Duplicate hospital-quarter records:", duplicate_count)

if duplicate_count > 0:
    raise ValueError("Duplicate hospital-quarter records must be investigated before matching quarters.")

In [ ]:
# Create a table containing the next quarter's target value.
target_next_quarter = df_model[
    ["FAC_NO", "PERIOD_NUMBER", "YEAR_QTR", "DIS_TOT"]
].copy()

target_next_quarter = target_next_quarter.rename(
    columns={
        "PERIOD_NUMBER": "TARGET_PERIOD_NUMBER",
        "YEAR_QTR": "TARGET_YEAR_QTR",
        "DIS_TOT": "DIS_TOT_NEXT_QTR"
    }
)

# For each previous-quarter record, identify the next consecutive quarter.
previous_quarter = df_model.copy()
previous_quarter["TARGET_PERIOD_NUMBER"] = (
    previous_quarter["PERIOD_NUMBER"] + 1
)

# Match each hospital's previous-quarter information to its next-quarter target.
model_df = previous_quarter.merge(
    target_next_quarter[
        ["FAC_NO", "TARGET_PERIOD_NUMBER", "TARGET_YEAR_QTR", "DIS_TOT_NEXT_QTR"]
    ],
    on=["FAC_NO", "TARGET_PERIOD_NUMBER"],
    how="inner"
)

# Keep rows with a known next-quarter target.
model_df = model_df.dropna(subset=["DIS_TOT_NEXT_QTR"]).copy()

# Rename the previous-quarter target so its timing is clear.
model_df = model_df.rename(
    columns={"DIS_TOT": "DIS_TOT_PREV_QTR"}
)

print("Future-quarter dataset shape:", model_df.shape)
print("Number of hospitals:", model_df["FAC_NO"].nunique())
print("Number of target quarters:", model_df["TARGET_YEAR_QTR"].nunique())

display(
    model_df[
        [
            "FAC_NO",
            "YEAR_QTR",
            "TARGET_YEAR_QTR",
            "DIS_TOT_PREV_QTR",
            "DIS_TOT_NEXT_QTR"
        ]
    ].head()
)

### Interpretation

The `YEAR_QTR` column identifies the quarter used for the predictor information. `TARGET_YEAR_QTR` identifies the following quarter being predicted.

For example, a row with `YEAR_QTR = 20254` and `TARGET_YEAR_QTR = 20261` uses 2025 Q4 information to predict 2026 Q1 discharges.

## 5. Select Candidate Features

The initial feature set is based on the EDA findings.

**Numerical features from the previous quarter:**
- `LIC_BEDS`
- `AVL_BEDS`
- `STF_BEDS`
- `DAY_TOT`
- `VIS_TOT`
- `DIS_TOT_PREV_QTR` (previous-quarter discharges)

**Categorical features from the previous quarter:**
- `TYPE_CNTRL`
- `TYPE_HOSP`
- `COUNTY_NAME`
- `HSA`

A bed availability ratio is also calculated from previous-quarter bed counts.

`TEACH_RURL` is not included in the initial feature set because the EDA found approximately 77.9% missing values. Discharge component variables are excluded because they are not needed for this forecasting design and may directly contribute to the discharge total.

The selected features are candidates for an initial model, not a final feature set.

In [ ]:
# Create a capacity feature using previous-quarter bed counts.
model_df["BED_AVAILABILITY_RATIO"] = np.where(
    model_df["LIC_BEDS"] > 0,
    model_df["AVL_BEDS"] / model_df["LIC_BEDS"],
    np.nan
)

# Numerical predictors from the previous quarter.
numeric_features = [
    "LIC_BEDS",
    "AVL_BEDS",
    "STF_BEDS",
    "DAY_TOT",
    "VIS_TOT",
    "DIS_TOT_PREV_QTR",
    "BED_AVAILABILITY_RATIO"
]

# Categorical predictors from the previous quarter.
categorical_features = [
    "TYPE_CNTRL",
    "TYPE_HOSP",
    "COUNTY_NAME",
    "HSA"
]

# Keep only features that are present in the dataset.
numeric_features = [
    feature for feature in numeric_features
    if feature in model_df.columns
]

categorical_features = [
    feature for feature in categorical_features
    if feature in model_df.columns
]

feature_columns = numeric_features + categorical_features
target_column = "DIS_TOT_NEXT_QTR"

print("Numerical features:")
for feature in numeric_features:
    print("-", feature)

print("\nCategorical features:")
for feature in categorical_features:
    print("-", feature)

print("\nTarget:", target_column)

## 6. Review Missing Values

Review missing values in the selected predictors and target.

Missing predictor values will be imputed later using statistics learned from the training data. Rows without a known target cannot be used for supervised model training and have already been excluded.

In [ ]:
# Summarize missing values in the selected predictors.
missing_summary = pd.DataFrame({
    "Missing_Count": model_df[feature_columns].isna().sum(),
    "Missing_Percent": model_df[feature_columns].isna().mean() * 100
}).sort_values("Missing_Percent", ascending=False)

display(missing_summary)

print("Missing target values:", model_df[target_column].isna().sum())

### Missing-value strategy

- Numerical predictors: replace missing values with the **training-set median**.
- Categorical predictors: replace missing values with the **training-set most frequent category**.
- Target (`DIS_TOT_NEXT_QTR`): do not impute; rows without a known target are excluded.

The imputation values will be learned from the training data only.

## 7. Split the Data Chronologically

Because this project predicts a future quarter, a random train/test split would not be appropriate. A chronological split better reflects how the model would be used in practice.

The target quarters are divided into:
- **Training set:** earliest 70% of target quarters.
- **Validation set:** following 15% of target quarters.
- **Testing set:** most recent 15% of target quarters.

All hospitals from the same target quarter remain in the same partition. The test set is kept separate for the final model evaluation.

In [ ]:
# Sort the unique target quarters chronologically.
target_periods = sorted(
    model_df["TARGET_PERIOD_NUMBER"].dropna().unique()
)

n_periods = len(target_periods)

if n_periods < 6:
    raise ValueError(
        "There are too few target quarters for a reliable chronological "
        "training, validation, and testing split."
    )

# Allocate the earliest periods to training, followed by validation and testing.
train_end = int(n_periods * 0.70)
validation_end = int(n_periods * 0.85)

# Ensure each partition has at least one target quarter.
train_end = max(1, train_end)
validation_end = max(train_end + 1, validation_end)
validation_end = min(validation_end, n_periods - 1)

train_periods = target_periods[:train_end]
validation_periods = target_periods[train_end:validation_end]
test_periods = target_periods[validation_end:]

# Split by target quarter, not by individual row.
train_df = model_df[
    model_df["TARGET_PERIOD_NUMBER"].isin(train_periods)
].copy()

validation_df = model_df[
    model_df["TARGET_PERIOD_NUMBER"].isin(validation_periods)
].copy()

test_df = model_df[
    model_df["TARGET_PERIOD_NUMBER"].isin(test_periods)
].copy()

print("Training target quarters:", len(train_periods))
print("Validation target quarters:", len(validation_periods))
print("Testing target quarters:", len(test_periods))

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows:", len(test_df))

print("\nTraining target periods:", sorted(train_df["TARGET_YEAR_QTR"].unique()))
print("Validation target periods:", sorted(validation_df["TARGET_YEAR_QTR"].unique()))
print("Testing target periods:", sorted(test_df["TARGET_YEAR_QTR"].unique()))

In [ ]:
# Confirm that the chronological partitions do not overlap.
train_set = set(train_periods)
validation_set = set(validation_periods)
test_set = set(test_periods)

assert train_set.isdisjoint(validation_set)
assert train_set.isdisjoint(test_set)
assert validation_set.isdisjoint(test_set)

# Confirm that the latest training period comes before validation,
# and validation comes before testing.
assert max(train_periods) < min(validation_periods)
assert max(validation_periods) < min(test_periods)

print("Chronological split checks passed.")

## 8. Separate Predictors and Target

Separate the selected predictor columns (`X`) from the next-quarter discharge target (`y`).

The target remains in its original units (number of discharges). Only predictor features are imputed, encoded, and scaled.

In [ ]:
# Training data.
X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

# Validation data.
X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[target_column].copy()

# Testing data.
X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("X_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_validation shape:", y_validation.shape)
print("y_test shape:", y_test.shape)

## 9. Build the Preprocessing Pipelines

Use a `ColumnTransformer` to apply the appropriate preprocessing to each feature type.

**Numerical pipeline:**
1. Impute missing values with the median.
2. Standardize features using `StandardScaler`.

**Categorical pipeline:**
1. Impute missing values with the most frequent category.
2. Convert categories into indicator (dummy) variables using `OneHotEncoder`.

The encoder is configured to ignore categories that may appear in validation or test data but were not present in the training data.

In [ ]:
# Preprocessing steps for numerical features.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing steps for categorical features.
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

# Apply the appropriate pipeline to each feature type.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    sparse_threshold=0
)

print("Preprocessing pipeline created.")

## 10. Fit on Training Data and Transform All Sets

Fit the preprocessing pipeline using **only** the training data.

Then use the fitted pipeline to transform the training, validation, and testing predictors. This prevents information from the validation and testing periods from influencing imputation, encoding, or scaling.

In [ ]:
# Learn imputation values, category levels, and scaling parameters from training data.
X_train_processed_array = preprocessor.fit_transform(X_train)

# Apply the fitted preprocessing steps to validation and testing data.
X_validation_processed_array = preprocessor.transform(X_validation)
X_test_processed_array = preprocessor.transform(X_test)

# Get the names of the transformed features.
processed_feature_names = preprocessor.get_feature_names_out()

# Convert the transformed arrays into DataFrames for easier inspection.
X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=processed_feature_names,
    index=X_train.index
)

X_validation_processed = pd.DataFrame(
    X_validation_processed_array,
    columns=processed_feature_names,
    index=X_validation.index
)

X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=processed_feature_names,
    index=X_test.index
)

print("Preprocessing completed.")
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_validation_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 11. Verify the Preprocessed Data

Check that:
- The transformed datasets have the same feature columns.
- Missing values have been handled.
- Numerical features have been standardized using training-set statistics.
- The target values remain separate from the predictors.

In [ ]:
# Display a sample of the transformed training data.
display(X_train_processed.head())

# Check missing values after preprocessing.
print("Missing values in training predictors:", X_train_processed.isna().sum().sum())
print("Missing values in validation predictors:", X_validation_processed.isna().sum().sum())
print("Missing values in testing predictors:", X_test_processed.isna().sum().sum())

# Confirm that all partitions have the same transformed columns.
assert list(X_train_processed.columns) == list(X_validation_processed.columns)
assert list(X_train_processed.columns) == list(X_test_processed.columns)

# Check the means and standard deviations of the standardized numeric features.
scaled_numeric_columns = [
    column for column in X_train_processed.columns
    if column.startswith("num__")
]

display(
    X_train_processed[scaled_numeric_columns]
    .agg(["mean", "std"])
    .T
)

print("Preprocessing verification completed.")

## 12. Save the Prepared Datasets and Preprocessing Pipeline

Save the transformed predictors, target values, and fitted preprocessing pipeline in the processed-data folder.

These files can be loaded during model development without fitting the preprocessing steps again.

The test set remains untouched for final model evaluation.

In [ ]:
# Folder for the prepared modeling datasets and preprocessing artifact.
OUTPUT_PATH = Path("../data/processed/modeling")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Save transformed predictors.
X_train_processed.to_csv(
    OUTPUT_PATH / "X_train_preprocessed.csv",
    index=False
)

X_validation_processed.to_csv(
    OUTPUT_PATH / "X_validation_preprocessed.csv",
    index=False
)

X_test_processed.to_csv(
    OUTPUT_PATH / "X_test_preprocessed.csv",
    index=False
)

# Save target values.
y_train.to_frame(name=target_column).to_csv(
    OUTPUT_PATH / "y_train.csv",
    index=False
)

y_validation.to_frame(name=target_column).to_csv(
    OUTPUT_PATH / "y_validation.csv",
    index=False
)

y_test.to_frame(name=target_column).to_csv(
    OUTPUT_PATH / "y_test.csv",
    index=False
)

# Save the fitted preprocessing pipeline.
joblib.dump(
    preprocessor,
    OUTPUT_PATH / "preprocessor.joblib"
)

print("Prepared datasets and preprocessing pipeline saved to:")
print(OUTPUT_PATH.resolve())

## 13. Summary

This notebook prepared the HCAI hospital data for a future-quarter discharge forecasting model.

The workflow:
- Matched each hospital's previous-quarter information with its next-quarter `DIS_TOT`.
- Excluded records without a consecutive-quarter match or a known target.
- Selected initial numerical and categorical predictors based on EDA findings.
- Created a bed availability ratio.
- Split target quarters chronologically into training, validation, and testing datasets.
- Imputed missing predictor values.
- Encoded categorical variables into indicator variables.
- Standardized numerical predictors.
- Saved the processed datasets and fitted preprocessing pipeline.

**Important modeling note:** The final test set should only be used for final model evaluation. Feature selection and model tuning should use the training and validation sets.

**Next step:** Develop and compare predictive models using the prepared training and validation datasets. Evaluate the selected model on the test set only after model selection is complete.